# T2I-RL Training with VLM Reward (Understanding-based)

**核心实验: 使用 VLM (GPT-4V/Claude) 作为奖励模型训练 T2I 生成**

这是项目的**主要实验**，使用 VLM 的理解能力来评估生成图像的质量。

## 实验设计

| 实验 | 奖励模型 | 目的 |
|------|---------|------|
| **主实验** | VLM (GPT-4V/Claude) | 项目核心 |
| 对比实验 | CLIP | 证明 VLM > CLIP |
| Baseline | 无训练 | 证明 RL 有效 |

## VLM 奖励优势

- **对象检测**: 检查图像中是否有正确的对象
- **属性验证**: 验证颜色、大小、形状是否正确
- **空间关系**: 理解 "on", "under", "next to" 等关系
- **数量计数**: 准确计数对象数量

---

## 0. 配置 API 密钥

**重要**: 运行前需要设置 API 密钥

In [ ]:
# ========================================
# API 配置 - 选择一个 VLM 提供商
# ========================================

# 选项 1: OpenAI GPT-4V (推荐)
OPENAI_API_KEY = "your-openai-api-key-here"  # 替换为你的 API key

# 选项 2: Anthropic Claude
ANTHROPIC_API_KEY = "your-anthropic-api-key-here"  # 替换为你的 API key

# 选择使用哪个 VLM
USE_VLM = "openai"  # "openai" 或 "anthropic"

# ========================================

import os
if USE_VLM == "openai":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("✓ Using OpenAI GPT-4V")
else:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
    print("✓ Using Anthropic Claude")

## 1. 安装依赖

In [ ]:
import time
start = time.time()

# Clean install
!pip uninstall -y transformers accelerate peft 2>/dev/null || true

# Install compatible versions
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.49.0 accelerate safetensors
!pip install -q --no-deps git+https://github.com/deepseek-ai/Janus.git
!pip install -q attrdict sentencepiece
!pip install -q open-clip-torch timm einops peft bitsandbytes
!pip install -q tqdm Pillow matplotlib wandb

# VLM API clients
!pip install -q openai anthropic

print(f"\n✓ Installation complete: {time.time()-start:.1f}s")
print("\n" + "="*50)
print("⚠️ RESTART RUNTIME NOW: Runtime -> Restart session")
print("="*50)

---
## ⚠️ 重启后从这里继续
---

In [ ]:
# 重新设置 API 密钥 (重启后需要重新运行)
import os

# ========================================
# 再次填入你的 API 密钥
# ========================================
OPENAI_API_KEY = "your-openai-api-key-here"
ANTHROPIC_API_KEY = "your-anthropic-api-key-here"
USE_VLM = "openai"  # "openai" 或 "anthropic"
# ========================================

if USE_VLM == "openai":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
else:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

print(f"✓ API Key configured for {USE_VLM}")

In [ ]:
# Clone and setup project
import os
import sys

if not os.path.exists('T2I-RL-Project'):
    !git clone https://github.com/Inoriany/T2I-RL-Project.git
else:
    %cd T2I-RL-Project
    !git pull
    %cd ..

os.chdir('T2I-RL-Project')
sys.path.insert(0, '.')
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Verify imports
import torch
import transformers
from janus.models import VLChatProcessor, MultiModalityCausalLM

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. 加载模型

In [ ]:
import torch
import gc
from src.models.generators import JanusProGenerator, GenerationConfig
from src.models.reward_models import VLMRewardModel, CLIPRewardModel

# Clear memory
gc.collect()
torch.cuda.empty_cache()

print("="*60)
print("Loading Janus-Pro-1B (bfloat16)")
print("="*60)

# Load generator
generator = JanusProGenerator(
    model_name_or_path="deepseek-ai/Janus-Pro-1B",
    device="cuda",
    dtype=torch.bfloat16,
)
generator.load_model()

print(f"\nGPU Memory after generator: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
print("="*60)
print("Loading VLM Reward Model")
print("="*60)

# Create VLM reward model
if USE_VLM == "openai":
    vlm_reward = VLMRewardModel(
        use_api=True,
        api_provider="openai",
        api_model="gpt-4o",  # or "gpt-4-vision-preview"
        device="cuda",
    )
else:
    vlm_reward = VLMRewardModel(
        use_api=True,
        api_provider="anthropic",
        api_model="claude-3-5-sonnet-20241022",
        device="cuda",
    )

print(f"✓ VLM Reward Model ready ({USE_VLM})")

# Also load CLIP for comparison
print("\nLoading CLIP for comparison...")
clip_reward = CLIPRewardModel(
    model_name="ViT-B-32",
    pretrained="openai",
    device="cuda",
)
clip_reward.load_model()
print("✓ CLIP Reward Model ready")

## 3. 测试 VLM 奖励

In [ ]:
import matplotlib.pyplot as plt

# Test prompts
test_prompts = [
    "a red apple on a blue plate",
    "two cats playing with a yellow ball",
]

config = GenerationConfig(
    num_inference_steps=30,
    guidance_scale=5.0,
    temperature=1.0,
    seed=42,
)

print("Generating test images...")
images = []
for prompt in test_prompts:
    img = generator.generate(prompt, config)
    images.extend(img)
    print(f"  ✓ Generated: {prompt}")

# Display images
fig, axes = plt.subplots(1, len(images), figsize=(6*len(images), 6))
for ax, img, prompt in zip(axes, images, test_prompts):
    ax.imshow(img)
    ax.set_title(prompt, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Compare VLM vs CLIP rewards
print("="*60)
print("Comparing VLM vs CLIP Rewards")
print("="*60)

# CLIP rewards
clip_out = clip_reward.compute_reward(images, test_prompts)
print("\nCLIP Rewards:")
for prompt, reward in zip(test_prompts, clip_out.rewards):
    print(f"  {prompt}: {reward.item():.4f}")

# VLM rewards (this calls the API)
print("\nVLM Rewards (calling API...):")
vlm_out = vlm_reward.compute_reward(images, test_prompts)
for prompt, reward in zip(test_prompts, vlm_out.rewards):
    print(f"  {prompt}: {reward.item():.4f}")

if hasattr(vlm_out, 'details') and vlm_out.details:
    print("\nVLM Details:")
    for i, detail in enumerate(vlm_out.details):
        print(f"  Image {i+1}: {detail}")

## 4. 训练配置

In [ ]:
from torch.utils.data import DataLoader
from src.data.dataset import PromptDataset
from src.training.grpo_trainer import GRPOTrainer, GRPOConfig
import json

# ========================================
# TRAINING CONFIG
# ========================================
NUM_PROMPTS = 100      # 用100个prompt (API费用考虑)
NUM_EPOCHS = 5         # 5个epochs
BATCH_SIZE = 2         # Batch size
SAVE_EVERY = 50        # 保存频率
USE_VLM_REWARD = True  # True=VLM奖励, False=CLIP奖励
# ========================================

# Load prompts
with open('data/prompts/train_prompts.json', 'r') as f:
    train_data = json.load(f)

all_prompts = []
for category, prompts in train_data['prompts'].items():
    all_prompts.extend(prompts)
    
train_prompts = all_prompts[:NUM_PROMPTS]
print(f"Training prompts: {len(train_prompts)}")

# Estimate cost
total_images = len(train_prompts) * NUM_EPOCHS * 2  # 2 samples per prompt
estimated_cost = total_images * 0.01  # ~$0.01 per image for GPT-4V
print(f"Estimated images to evaluate: {total_images}")
print(f"Estimated API cost: ${estimated_cost:.2f}")

# Create dataset
train_dataset = PromptDataset(train_prompts)
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

# Training config
grpo_config = GRPOConfig(
    learning_rate=1e-5,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    num_samples_per_prompt=2,
    gradient_accumulation_steps=4,
    kl_coef=0.1,
    clip_range=0.2,
    max_grad_norm=1.0,
    warmup_steps=20,
    use_wandb=False,
    output_dir="./outputs/grpo_vlm" if USE_VLM_REWARD else "./outputs/grpo_clip",
    save_steps=SAVE_EVERY,
    log_steps=10,
)

print(f"\nReward Model: {'VLM' if USE_VLM_REWARD else 'CLIP'}")
print(f"Output dir: {grpo_config.output_dir}")

In [ ]:
# Enable LoRA
print("Enabling LoRA...")
generator.enable_lora(lora_config={
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
})

trainable = sum(p.numel() for p in generator.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in generator.model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Create trainer with selected reward model
reward_model = vlm_reward if USE_VLM_REWARD else clip_reward

trainer = GRPOTrainer(
    generator=generator,
    reward_model=reward_model,
    config=grpo_config,
    train_dataloader=train_dataloader,
)

print(f"✓ Trainer created with {'VLM' if USE_VLM_REWARD else 'CLIP'} reward")

## 5. 运行训练

In [ ]:
import time
from tqdm.auto import tqdm

print("="*60)
print(f"Starting GRPO Training with {'VLM' if USE_VLM_REWARD else 'CLIP'} Reward")
print("="*60)

start_time = time.time()
step = 0
total_loss = 0
total_reward = 0
log_steps = grpo_config.log_steps

# Training history for plotting
history = {'step': [], 'loss': [], 'reward': []}

for epoch in range(grpo_config.num_epochs):
    print(f"\nEpoch {epoch + 1}/{grpo_config.num_epochs}")
    
    progress = tqdm(train_dataloader, desc="Training")
    for batch in progress:
        try:
            # Compute loss
            loss_dict = trainer.compute_loss(batch)
            loss = loss_dict['total_loss']
            
            # Backward
            loss.backward()
            
            # Gradient accumulation
            if (step + 1) % grpo_config.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(
                    generator.model.parameters(), 
                    grpo_config.max_grad_norm
                )
                trainer.optimizer.step()
                trainer.optimizer.zero_grad()
            
            # Logging
            total_loss += loss.item()
            if 'mean_reward' in loss_dict:
                total_reward += loss_dict['mean_reward']
            
            step += 1
            
            if step % log_steps == 0:
                avg_loss = total_loss / log_steps
                avg_reward = total_reward / log_steps
                
                history['step'].append(step)
                history['loss'].append(avg_loss)
                history['reward'].append(avg_reward)
                
                progress.set_postfix({
                    'loss': f'{avg_loss:.4f}',
                    'reward': f'{avg_reward:.4f}',
                    'mem': f'{torch.cuda.memory_allocated()/1e9:.1f}GB'
                })
                total_loss = 0
                total_reward = 0
            
            # Save checkpoint
            if step % grpo_config.save_steps == 0:
                save_path = f"{grpo_config.output_dir}/checkpoint-{step}"
                os.makedirs(save_path, exist_ok=True)
                generator.save_lora(save_path)
                print(f"\n✓ Saved checkpoint: {save_path}")
                
        except Exception as e:
            if "out of memory" in str(e).lower():
                print(f"\nOOM at step {step}, clearing cache...")
                torch.cuda.empty_cache()
                gc.collect()
                continue
            else:
                print(f"\nError at step {step}: {e}")
                raise e

elapsed = time.time() - start_time
print(f"\n" + "="*60)
print(f"✓ Training complete!")
print(f"  Total steps: {step}")
print(f"  Time: {elapsed/60:.1f} minutes")
print("="*60)

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['step'], history['loss'])
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')

ax2.plot(history['step'], history['reward'])
ax2.set_xlabel('Step')
ax2.set_ylabel('Reward')
ax2.set_title('Mean Reward')

plt.tight_layout()
plt.show()

## 6. 保存模型

In [ ]:
# Save final model
final_path = f"{grpo_config.output_dir}/final"
os.makedirs(final_path, exist_ok=True)
generator.save_lora(final_path)
print(f"✓ Saved final model to {final_path}")

# Save training history
import json
with open(f"{grpo_config.output_dir}/history.json", 'w') as f:
    json.dump(history, f)
print(f"✓ Saved training history")

# Zip for download
!zip -r outputs_vlm.zip outputs/
print("\n📦 Download outputs_vlm.zip to save your trained model!")

## 7. 评估训练结果

In [ ]:
# Evaluation prompts (different from training)
eval_prompts = [
    "a red apple on a wooden table",
    "a blue car next to a yellow house",
    "three birds sitting on a wire",
    "a large elephant and a small mouse",
    "a cat sleeping under a chair",
]

print("="*60)
print("Evaluation: Generating images with trained model")
print("="*60)

eval_config = GenerationConfig(
    num_inference_steps=50,
    guidance_scale=5.0,
    temperature=1.0,
    seed=123,
)

eval_images = []
for prompt in eval_prompts:
    img = generator.generate(prompt, eval_config)
    eval_images.extend(img)
    print(f"  ✓ {prompt}")

In [ ]:
# Display evaluation images
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flat

for i, (ax, img, prompt) in enumerate(zip(axes, eval_images, eval_prompts)):
    ax.imshow(img)
    ax.set_title(prompt, fontsize=10)
    ax.axis('off')

# Hide empty subplot
if len(eval_images) < 6:
    axes[5].axis('off')
    
plt.tight_layout()
plt.show()

In [ ]:
# Compare rewards
print("="*60)
print("Evaluation Rewards")
print("="*60)

# CLIP rewards
eval_clip = clip_reward.compute_reward(eval_images, eval_prompts)

# VLM rewards
print("\nCalling VLM API for evaluation...")
eval_vlm = vlm_reward.compute_reward(eval_images, eval_prompts)

print("\n" + "-"*60)
print(f"{'Prompt':<45} {'CLIP':>8} {'VLM':>8}")
print("-"*60)
for i, prompt in enumerate(eval_prompts):
    clip_r = eval_clip.rewards[i].item()
    vlm_r = eval_vlm.rewards[i].item()
    print(f"{prompt:<45} {clip_r:>8.4f} {vlm_r:>8.4f}")

print("-"*60)
print(f"{'Mean':<45} {eval_clip.rewards.mean().item():>8.4f} {eval_vlm.rewards.mean().item():>8.4f}")

## 8. 费用和内存报告

In [ ]:
print("="*60)
print("Final Report")
print("="*60)

# GPU Memory
print("\nGPU Memory:")
print(f"  Current: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"  Peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

# Training stats
print(f"\nTraining:")
print(f"  Total steps: {step}")
print(f"  Time: {elapsed/60:.1f} minutes")
print(f"  Final loss: {history['loss'][-1]:.4f}" if history['loss'] else "  N/A")
print(f"  Final reward: {history['reward'][-1]:.4f}" if history['reward'] else "  N/A")

# Estimated cost
actual_images = step * 2  # 2 samples per step
actual_cost = actual_images * 0.01
print(f"\nEstimated API Cost:")
print(f"  Images evaluated: {actual_images}")
print(f"  Estimated cost: ${actual_cost:.2f}")

---
## Done!

你的 VLM 奖励训练完成了！

**保存的文件:**
- `outputs/grpo_vlm/final/` - 最终 LoRA 权重
- `outputs/grpo_vlm/history.json` - 训练曲线数据
- `outputs_vlm.zip` - 打包下载

**下一步:**
1. 下载 `outputs_vlm.zip`
2. 运行 CLIP 版本作为对比 (设置 `USE_VLM_REWARD = False`)
3. 运行 Benchmark 评估
4. 写报告比较 VLM vs CLIP 结果